# Convexity and Convergence Guarantees

| Difficulty | █████░░░░░ 5/10 |
| :--- | :--- |
| Prerequisites | 04.02 |
| Time | ~65 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/04_Optimization/03_convexity_and_convergence.ipynb)

---

## 🎯 Learning Objective

Define convex functions and sets, and understand what convergence guarantees convexity buys an optimizer.

---

## 📐 Theory

`04.01` and `04.02` treated non-convexity as the default. This notebook studies the special case
where it's absent — because when a loss *is* convex, gradient descent's behavior becomes
provably predictable rather than merely empirically reasonable.

### Convex sets and functions

A set $C$ is **convex** if the line segment between any two points in $C$ stays inside $C$:
$\theta \mathbf{x}+(1-\theta)\mathbf{y} \in C$ for all $\mathbf{x},\mathbf{y}\in C$,
$\theta\in[0,1]$. A function $f$ is **convex** if its epigraph is convex, equivalently if it
lies below every chord connecting two of its points:

$$f(\theta \mathbf{x}+(1-\theta)\mathbf{y}) \le \theta f(\mathbf{x}) + (1-\theta) f(\mathbf{y})
\quad \forall\, \mathbf{x},\mathbf{y},\ \theta\in[0,1]$$

For twice-differentiable $f$, this is equivalent to the Hessian (`02.06`) being positive
semi-definite everywhere: $H(\mathbf{x}) \succeq 0\ \forall \mathbf{x}$. The payoff is enormous:
a convex function's **every local minimum is global**, and there are no spurious local minima
or saddle points to worry about — the exact opposite of `04.01`'s landscape.

### Strong convexity: a quantitative lower bound on curvature

Convexity says the Hessian's eigenvalues are $\ge 0$; **$\mu$-strong convexity** demands they're
bounded *away* from zero: $H(\mathbf{x}) \succeq \mu I$ for some $\mu>0$. Geometrically, $f$
curves upward at least as fast as the bowl $\frac{\mu}{2}\|\mathbf{x}\|^2$ in every direction.
This rules out the flat-bottomed case ($f(x)=x^4$ is convex but not strongly convex — its
Hessian is $0$ at the origin) and, crucially, guarantees a *unique* minimizer.

### Smoothness (Lipschitz gradients): an upper bound on curvature

The complementary condition, **$L$-smoothness**, bounds curvature from above:
$\|\nabla f(\mathbf{x}) - \nabla f(\mathbf{y})\| \le L\|\mathbf{x}-\mathbf{y}\|$ — the gradient
can't change arbitrarily fast, equivalently $H(\mathbf{x}) \preceq LI$. Together, $\mu$-strong
convexity and $L$-smoothness sandwich every Hessian eigenvalue in $[\mu, L]$.

### Convergence rate: the condition number decides everything

For an $L$-smooth, $\mu$-strongly convex $f$, gradient descent with the optimal constant step
size $\eta=\frac{2}{L+\mu}$ converges **linearly**: $f(\mathbf{w}_k)-f^* \le \rho^k
\left(f(\mathbf{w}_0)-f^*\right)$ with rate $\rho=\left(\frac{L-\mu}{L+\mu}\right)^2$ (Nesterov,
2004; Boyd & Vandenberghe, 2004). Writing $\kappa=L/\mu$ (the **condition number** — recall
`01.06`'s connection between eigenvalue spread and matrix conditioning), $\rho$ approaches $1$
as $\kappa\to\infty$: an ill-conditioned bowl (elongated, like `04.02`'s correlated-feature
example) forces painfully slow convergence, while $\kappa\approx 1$ (nearly spherical contours)
converges almost immediately. This single number is *the* quantity that separates "easy" convex
optimization from "hard" convex optimization — non-convexity isn't the only source of
difficulty.

### Logistic regression: convex, in practice

Linear/logistic regression losses are convex (verified below), so training one has none of
`04.01`'s landscape hazards: gradient descent provably finds *the* global minimum, given enough
steps and a small enough $\eta$. Deep networks lose this guarantee entirely — their loss is a
composition of convex pieces through non-convex nonlinearities — which is precisely why Module
`04` needs the extra machinery (momentum, adaptive rates, second-order methods) that a convex
problem never requires.

### The convergence hierarchy, derived

The Theory section above *stated* rates; this subsection *proves* them. The key
ingredients are strict convexity, the descent lemma, and the interplay between
strong convexity and smoothness.

#### 1. Strict convexity

A function $f$ is **strictly convex** if the chord inequality holds with
*strict* inequality for distinct points:

$$f(\theta \mathbf{x}+(1-\theta)\mathbf{y}) < \theta f(\mathbf{x}) + (1-\theta)f(\mathbf{y})
\quad \forall\, \mathbf{x}\neq\mathbf{y},\ \theta\in(0,1)$$

**Consequence — unique minimizer.** Suppose $f$ has two distinct minimizers
$\mathbf{a}\neq\mathbf{b}$ with $f(\mathbf{a})=f(\mathbf{b})=f^*$. Their midpoint
satisfies $f\!\bigl(\tfrac12\mathbf{a}+\tfrac12\mathbf{b}\bigr) < \tfrac12 f(\mathbf{a})
+\tfrac12 f(\mathbf{b})=f^*$, contradicting both being minimal. $\square$

**Hierarchy.** Strong convexity $\Longrightarrow$ strict convexity
$\Longrightarrow$ convexity (not conversely).
The function $f(x)=x^4$ is *strictly* but *not strongly* convex: every chord
inequality is strict (the function has no flat segments), yet $f''(0)=0$,
so no $\mu>0$ satisfies $f''(x)\ge\mu$ everywhere. This intermediate notion
guarantees a unique minimizer **without** a quantitative convergence rate —
strong convexity is what buys the rate.

#### 2. Lipschitz continuity of $f$

$f$ is **$G$-Lipschitz** if its *values* cannot change faster than linearly:

$$|f(\mathbf{x})-f(\mathbf{y})| \le G\|\mathbf{x}-\mathbf{y}\| \quad \forall\,\mathbf{x},\mathbf{y}$$

This is **not** the same as $L$-smoothness (which bounds the *gradient's*
rate of change: $\|\nabla f(\mathbf{x})-\nabla f(\mathbf{y})\|\le
L\|\mathbf{x}-\mathbf{y}\|$). The absolute-value function $f(x)=|x|$ is
$1$-Lipschitz (slopes are $\pm1$) but **not** smooth — the gradient doesn't
even exist at $x=0$. Conversely, a smooth quadratic $f(x)=\tfrac{L}{2}x^2$
has Lipschitz gradient but is *not* globally Lipschitz in $f$ (it grows
without bound). The distinction matters because non-smooth but Lipschitz
functions can still be optimised via *subgradient* methods (`04.10`), just
more slowly.

#### 3. The descent lemma

For an $L$-smooth function $f$:

$$\boxed{f(\mathbf{y}) \le f(\mathbf{x}) + \nabla f(\mathbf{x})^{\!\top}(\mathbf{y}-\mathbf{x})
+ \frac{L}{2}\|\mathbf{y}-\mathbf{x}\|^2} \qquad \text{(descent lemma)}$$

*Proof sketch.* By the fundamental theorem of calculus,
$f(\mathbf{y})-f(\mathbf{x})=\int_0^1 \nabla f(\mathbf{x}+t(\mathbf{y}-\mathbf{x}))^{\!\top}
(\mathbf{y}-\mathbf{x})\,dt$. Adding and subtracting $\nabla f(\mathbf{x})^{\!\top}
(\mathbf{y}-\mathbf{x})$ and applying the $L$-smoothness bound
$\|\nabla f(\mathbf{x}+t(\mathbf{y}-\mathbf{x}))-\nabla f(\mathbf{x})\|
\le Lt\|\mathbf{y}-\mathbf{x}\|$ inside the integral gives the result after
evaluating $\int_0^1 t\,dt=\tfrac12$. $\square$

Geometrically, $L$-smoothness guarantees that the function lies below a
**quadratic upper bound** centred at every point — the descent lemma is that
upper bound written explicitly.

#### 4. Proof: GD converges at $O(1/T)$ for convex, $L$-smooth $f$

Set $\mathbf{y}=\mathbf{w}_{t+1}=\mathbf{w}_t - \tfrac{1}{L}\nabla f(\mathbf{w}_t)$
and $\mathbf{x}=\mathbf{w}_t$ in the descent lemma:

$$f(\mathbf{w}_{t+1}) \le f(\mathbf{w}_t) - \frac{1}{2L}\|\nabla f(\mathbf{w}_t)\|^2$$

By convexity, $f(\mathbf{w}_t)-f^* \le \nabla f(\mathbf{w}_t)^{\!\top}
(\mathbf{w}_t - \mathbf{w}^*)$, and Cauchy–Schwarz plus $\|\nabla f\|^2$
give, after telescoping the sum $\sum_{t=0}^{T-1}$ and using the minimum
over all iterates:

$$\boxed{f(\mathbf{w}_T)-f^* \le \frac{L\|\mathbf{w}_0-\mathbf{w}^*\|^2}{2T}}$$

Halving the error requires **doubling** the iterations — *sub*-linear
convergence. This is the price of convexity without strong convexity.

#### 5. Proof: GD converges at $O(\rho^T)$ for $\mu$-strongly-convex, $L$-smooth $f$

Strong convexity gives the complementary bound
$f(\mathbf{x})-f^* \ge \frac{\mu}{2}\|\mathbf{x}-\mathbf{w}^*\|^2$, so
$\|\nabla f(\mathbf{w}_t)\|^2 \ge 2\mu(f(\mathbf{w}_t)-f^*)$. Substituting
into the per-step descent inequality from (4):

$$f(\mathbf{w}_{t+1})-f^* \le \Bigl(1-\frac{\mu}{L}\Bigr)\bigl(f(\mathbf{w}_t)-f^*\bigr)$$

Setting $\rho = 1-\frac{1}{\kappa}$ where $\kappa=L/\mu$:

$$\boxed{f(\mathbf{w}_T)-f^* \le \rho^T\bigl(f(\mathbf{w}_0)-f^*\bigr),
\qquad \rho = 1 - \frac{1}{\kappa}}$$

This is **linear** (geometric) convergence — the error shrinks by a constant
factor every step. A larger condition number $\kappa$ pushes $\rho$ closer to
$1$, slowing convergence, exactly as the earlier subsection predicted.

> *Note.* The optimal step size $\eta^*=2/(L+\mu)$ from the “Convergence
> rate” section gives the tighter contraction $\rho=((\kappa-1)/(\kappa+1))^2$
> (proved in Exercise 3). The $\eta=1/L$ analysis here trades a slightly
> looser bound for a simpler proof.

#### 6. Norms and dual norms

For a norm $\|\cdot\|$, its **dual norm** is:

$$\|\mathbf{y}\|_* = \sup_{\|\mathbf{x}\|\le 1}\langle \mathbf{y},\mathbf{x}\rangle$$

The **generalised Cauchy–Schwarz inequality** follows immediately:
$\langle \mathbf{x},\mathbf{y}\rangle \le \|\mathbf{x}\|\cdot\|\mathbf{y}\|_*$.
Key examples:

| Norm | Dual |
|:---|:---|
| $\ell_2$ | $\ell_2$ (self-dual) |
| $\ell_1$ | $\ell_\infty$ |
| $\ell_\infty$ | $\ell_1$ |

**Why this matters for ML.** The optimality condition for
$\min_{\mathbf{w}} f(\mathbf{w})+\lambda\|\mathbf{w}\|_1$ is
$0\in\nabla f(\mathbf{w}^*)+\lambda\,\partial\|\mathbf{w}^*\|_1$, which
implies $\|\nabla f(\mathbf{w}^*)\|_\infty \le \lambda$ — i.e. the gradient
in the **dual norm** must be small. Every coordinate $j$ with
$|\partial_j f(\mathbf{w}^*)| < \lambda$ is forced to $w_j^*=0$ (exact
sparsity). This is precisely why $\ell_1$ regularisation produces sparse
solutions — it is a statement about dual norms, not about “corners” of the
$\ell_1$ ball (a common but imprecise folk explanation).

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

The chord test makes convexity visible directly, and plotting gradient descent's theoretical
convergence rate against the condition number $\kappa$ shows exactly why some convex problems
are still much harder than others.

In [ ]:
# Left two panels: the chord test (definition of convexity) applied to a convex and a
# non-convex function. Right panel: gradient descent's theoretical linear-convergence rate
# rho = ((kappa-1)/(kappa+1))^2 as a function of the condition number kappa=L/mu.
def f_convex(x):
    return x ** 2

def f_nonconvex(x):
    return -2 * x**2 + x**4

x_range = np.linspace(-1.8, 1.8, 200)
x1, x2 = -1.5, 1.2

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, f, name in [(axes[0], f_convex, "Convex: f(x)=x^2"),
                     (axes[1], f_nonconvex, "Non-convex: f(x)=-2x^2+x^4")]:
    ax.plot(x_range, f(x_range), color="#4C72B0", lw=2)
    chord_x = np.linspace(x1, x2, 50)
    chord_y = f(x1) + (f(x2) - f(x1)) * (chord_x - x1) / (x2 - x1)
    ax.plot(chord_x, chord_y, "--", color="#C44E52", lw=1.5, label="chord")
    ax.plot([x1, x2], [f(x1), f(x2)], "ko")
    below_chord = np.all(f(chord_x) <= chord_y + 1e-9)
    ax.set_title(f"{name}\nf always <= chord: {below_chord}")
    ax.legend(fontsize=8)

kappas = np.logspace(0, 3, 200)
rhos = ((kappas - 1) / (kappas + 1)) ** 2
axes[2].semilogx(kappas, rhos, color="#55A868", lw=2)
axes[2].set_xlabel("condition number kappa = L/mu"); axes[2].set_ylabel("convergence rate rho")
axes[2].set_title("Gradient descent's rate vs conditioning:\nkappa near 1 is fast, large kappa stalls")
plt.tight_layout()
plt.show()

print("Chord-test violation counts over 1000 random point pairs:")
rng = np.random.default_rng(0)
for f, name in [(f_convex, "f(x)=x^2"), (f_nonconvex, "f(x)=-2x^2+x^4")]:
    violations = 0
    for _ in range(1000):
        a, b = rng.uniform(-1.5, 1.5, 2)
        theta = rng.uniform(0, 1)
        if f(theta * a + (1 - theta) * b) > theta * f(a) + (1 - theta) * f(b) + 1e-9:
            violations += 1
    print(f"  {name}: {violations}/1000 violations")

## 🐍 Implementation from Scratch

We run gradient descent on quadratics with different condition numbers $\kappa$, measure the
*empirical* per-step shrinkage factor of the optimality gap, and check it against the
theoretical rate $\rho=\left(\frac{\kappa-1}{\kappa+1}\right)^2$ from the Theory section. Then we
verify logistic regression's loss is convex by checking its Hessian is positive semi-definite
across many random points, cross-checked against `scipy.optimize`.

In [ ]:
from scipy.optimize import minimize

def gd_on_quadratic(mu, L, steps=200):
    """Gradient descent on f(x)=0.5*x^T diag(mu,L) x, with the optimal constant step size
    eta=2/(L+mu) from the Theory section. Returns the optimality gap f(x_k)-f* at every step."""
    A = np.diag([mu, L])
    x = np.array([1.0, 1.0])
    eta = 2.0 / (L + mu)
    gaps = []
    for _ in range(steps):
        gaps.append(0.5 * x @ A @ x)   # f* = 0 here, so the gap IS the loss
        x = x - eta * (A @ x)
    return np.array(gaps)

print("Empirical vs theoretical convergence rate rho, for several condition numbers:")
for mu, L in [(1, 1), (1, 4), (1, 10), (1, 50)]:
    gaps = gd_on_quadratic(mu, L)
    kappa = L / mu
    theoretical_rho = ((kappa - 1) / (kappa + 1)) ** 2
    empirical_rho = (gaps[100:150] / gaps[99:149]).mean() if kappa > 1 else float("nan")
    print(f"  kappa={kappa:5.1f}:  theoretical rho={theoretical_rho:.5f}   empirical rho={empirical_rho:.5f}")

# ---------- Verify logistic regression's loss is convex ----------
rng = np.random.default_rng(0)
N, d = 200, 3
X = rng.normal(size=(N, d))
true_w = np.array([1.5, -2.0, 0.5])
y = (rng.uniform(size=N) < 1 / (1 + np.exp(-X @ true_w))).astype(float)

def logistic_loss(w, X, y):
    z = X @ w
    return np.mean(np.log1p(np.exp(-np.abs(z))) + np.maximum(z, 0) - y * z)  # stable log(1+e^z)-yz

def logistic_grad(w, X, y):
    p = 1 / (1 + np.exp(-X @ w))
    return X.T @ (p - y) / len(y)

def logistic_hessian(w, X, y):
    p = 1 / (1 + np.exp(-X @ w))
    return (X * (p * (1 - p))[:, None]).T @ X / len(y)  # X^T diag(p(1-p)) X, always PSD

min_eigval = min(np.linalg.eigvalsh(logistic_hessian(rng.normal(size=d) * 2, X, y)).min() for _ in range(200))
print(f"\nSmallest Hessian eigenvalue over 200 random weight vectors: {min_eigval:.6f} (>= 0 means convex)")

# Cross-check our gradient descent solution against scipy's independent BFGS solver
w_gd = np.zeros(d)
for _ in range(2000):
    w_gd = w_gd - 0.5 * logistic_grad(w_gd, X, y)
result = minimize(logistic_loss, np.zeros(d), args=(X, y), jac=logistic_grad, method="BFGS")
print(f"our gradient descent: w={np.round(w_gd, 4)}, loss={logistic_loss(w_gd, X, y):.5f}")
print(f"scipy BFGS:           w={np.round(result.x, 4)}, loss={result.fun:.5f}")
print(f"match: {np.allclose(w_gd, result.x, atol=1e-3)}")

In [ ]:
# ---------- Verify the descent lemma empirically on a quadratic ----------
# For f(x) = 0.5 * x^T A x (a quadratic), the descent lemma states:
#   f(y) <= f(x) + grad_f(x)^T (y-x) + (L/2)||y-x||^2
# where L is the largest eigenvalue of A.

rng_dl = np.random.default_rng(42)
d_dl = 5
M = rng_dl.normal(size=(d_dl, d_dl))
A_dl = M.T @ M + 0.5 * np.eye(d_dl)          # PSD with eigenvalues in [0.5, ~large]
eigvals = np.linalg.eigvalsh(A_dl)
mu_dl, L_dl = eigvals.min(), eigvals.max()
print(f"Quadratic: mu={mu_dl:.3f}, L={L_dl:.3f}, kappa={L_dl/mu_dl:.1f}")

def f_quad(x): return 0.5 * x @ A_dl @ x
def grad_quad(x): return A_dl @ x

violations = 0
n_tests = 5000
for _ in range(n_tests):
    x_test = rng_dl.normal(size=d_dl) * 3
    y_test = rng_dl.normal(size=d_dl) * 3
    lhs = f_quad(y_test)
    rhs = f_quad(x_test) + grad_quad(x_test) @ (y_test - x_test) + (L_dl / 2) * np.sum((y_test - x_test)**2)
    if lhs > rhs + 1e-10:
        violations += 1
print(f"Descent lemma violations: {violations}/{n_tests} (expect 0)")

# ---------- Convergence comparison: O(1/T) vs O(rho^T) ----------
# Problem A: convex + smooth (but NOT strongly convex) — f(w) = sum of softplus
# Problem B: strongly-convex + smooth — f(w) = 0.5 * w^T A w with mu > 0

# Problem A: Use a barely-convex function: f(w) = (1/d) sum_i log(1 + exp(w_i))
# This is convex, L-smooth with L=0.25 (max second derivative of logistic sigmoid),
# but NOT strongly convex (Hessian eigenvalues approach 0 as |w| -> inf).
# For a clean comparison, use a quadratic with tiny mu vs large mu.

# Convex-only: eigenvalues in [1e-4, 10] — nearly flat direction makes it "barely" SC
# Strongly-convex: eigenvalues in [1, 10]
steps_conv = 500

def run_gd_general(A, steps):
    """GD on f(x) = 0.5 x^T A x with eta=1/L, return gaps."""
    eigs = np.linalg.eigvalsh(A)
    L_loc = eigs.max()
    eta = 1.0 / L_loc
    x = np.ones(A.shape[0])
    gaps = []
    for _ in range(steps):
        gaps.append(0.5 * x @ A @ x)
        x = x - eta * (A @ x)
    return np.array(gaps)

# Convex (barely SC): mu ~ 0.001
A_convex = np.diag([0.001, 0.5, 2.0, 5.0, 10.0])
gaps_convex = run_gd_general(A_convex, steps_conv)

# Strongly convex: mu = 1
A_sc = np.diag([1.0, 2.0, 4.0, 7.0, 10.0])
gaps_sc = run_gd_general(A_sc, steps_conv)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: log-scale convergence
t = np.arange(1, steps_conv + 1)
axes[0].semilogy(t, gaps_convex, color='#C44E52', lw=2, label=r'Convex ($\mu\approx0$): $O(1/T)$')
axes[0].semilogy(t, gaps_sc, color='#4C72B0', lw=2, label=r'Strongly convex ($\mu=1$): $O(\rho^T)$')
# Overlay theoretical envelopes
L_c = 10.0; R2_c = np.sum(np.ones(5)**2)
axes[0].semilogy(t, L_c * R2_c / (2 * t), '--', color='#C44E52', alpha=0.6, label=r'$LR^2/(2T)$ bound')
kappa_sc = 10.0 / 1.0
rho_sc = 1 - 1.0 / kappa_sc
axes[0].semilogy(t, gaps_sc[0] * rho_sc**t, '--', color='#4C72B0', alpha=0.6, label=r'$\rho^T$ bound ($\rho=1-1/\kappa$)')
axes[0].set_xlabel('Iteration $T$'); axes[0].set_ylabel(r'$f(\mathbf{w}_T) - f^*$')
axes[0].set_title('Convergence: sublinear vs linear')
axes[0].legend(fontsize=8); axes[0].set_ylim(bottom=1e-15)

# Right: log-log to see the 1/T slope
axes[1].loglog(t, gaps_convex, color='#C44E52', lw=2, label=r'Convex ($\mu\approx0$)')
axes[1].loglog(t, L_c * R2_c / (2 * t), '--', color='gray', alpha=0.7, label=r'$\propto 1/T$ reference')
axes[1].set_xlabel('Iteration $T$ (log scale)'); axes[1].set_ylabel(r'$f(\mathbf{w}_T) - f^*$ (log scale)')
axes[1].set_title(r'Log-log confirms $O(1/T)$ slope for convex case')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

print(f"\nStrongly-convex empirical rho (late window): "
      f"{(gaps_sc[200:250] / gaps_sc[199:249]).mean():.6f}")
print(f"Theoretical rho = 1 - 1/kappa = {rho_sc:.6f}")

## 🤖 Why This Matters for AI

Logistic regression is the textbook example of convex ML: its loss (cross-entropy) has a
positive-semi-definite Hessian everywhere, so gradient descent provably converges to *the*
global minimum regardless of where it starts. Swap the linear model for even a tiny neural
network and that guarantee vanishes — the composition of a convex loss with a non-convex
feature map (the hidden layer) is generally non-convex, so different random initializations can
land in different, non-identical solutions, exactly as `04.01` predicted. We make that contrast
concrete below: run both models from many wildly different random starting points and compare
how much the final solutions actually differ.

In [ ]:
# Same classification task, two models: logistic regression (convex) vs a small 1-hidden-layer
# network (non-convex, since a tanh hidden layer composed with a convex loss is not convex).
# A touch of nonlinearity in the true decision boundary means neither model fits perfectly --
# what we care about is how much the LEARNED solution varies across random restarts.
rng_ai = np.random.default_rng(7)
N_ai, d_ai = 200, 3
X_ai = rng_ai.normal(size=(N_ai, d_ai))
z_true = X_ai @ np.array([1.5, -2.0, 0.5]) + 0.3 * np.sin(3 * X_ai[:, 0])
y_ai = (rng_ai.uniform(size=N_ai) < 1 / (1 + np.exp(-z_true))).astype(float)

def logistic_loss_ai(w, X, y):
    z = X @ w
    return np.mean(np.log1p(np.exp(-np.abs(z))) + np.maximum(z, 0) - y * z)

def logistic_grad_ai(w, X, y):
    p = 1 / (1 + np.exp(-X @ w))
    return X.T @ (p - y) / len(y)

def mlp_loss_grad(w_flat, X, y, d_hidden):
    i1 = d_ai * d_hidden
    W1 = w_flat[:i1].reshape(d_ai, d_hidden)
    b1 = w_flat[i1:i1 + d_hidden]
    W2 = w_flat[i1 + d_hidden:i1 + 2 * d_hidden].reshape(d_hidden, 1)
    b2 = w_flat[-1]
    A1 = np.tanh(X @ W1 + b1)
    z = (A1 @ W2 + b2).ravel()
    p = 1 / (1 + np.exp(-z))
    loss = np.mean(np.log1p(np.exp(-np.abs(z))) + np.maximum(z, 0) - y * z)
    n = len(y)
    dz = (p - y) / n
    dW2 = A1.T @ dz[:, None]
    db2 = dz.sum()
    dZ1 = (dz[:, None] @ W2.reshape(1, -1)) * (1 - A1 ** 2)
    dW1 = X.T @ dZ1
    db1 = dZ1.sum(axis=0)
    return loss, np.concatenate([dW1.ravel(), db1.ravel(), dW2.ravel(), [db2]])

d_hidden = 6
n_params = d_ai * d_hidden + d_hidden + d_hidden + 1
n_restarts = 8

logistic_finals, mlp_finals = [], []
for seed in range(n_restarts):
    r = np.random.default_rng(seed)
    w_log = r.normal(scale=3.0, size=d_ai)
    for _ in range(1500):
        w_log = w_log - 0.3 * logistic_grad_ai(w_log, X_ai, y_ai)
    logistic_finals.append(logistic_loss_ai(w_log, X_ai, y_ai))

    w_mlp = r.normal(scale=3.0, size=n_params)
    for _ in range(1500):
        loss, grad = mlp_loss_grad(w_mlp, X_ai, y_ai, d_hidden)
        w_mlp = w_mlp - 0.3 * grad
    mlp_finals.append(loss)

print(f"{'model':22s} {'min loss':>9} {'max loss':>9} {'spread':>8}")
print(f"{'logistic (convex)':22s} {min(logistic_finals):9.6f} {max(logistic_finals):9.6f} "
      f"{max(logistic_finals)-min(logistic_finals):8.6f}")
print(f"{'1-hidden MLP (not)':22s} {min(mlp_finals):9.6f} {max(mlp_finals):9.6f} "
      f"{max(mlp_finals)-min(mlp_finals):8.6f}")
print(f"\n{n_restarts} wildly different random restarts: logistic regression lands on the SAME")
print("solution every time (spread is numerical noise); the MLP lands on genuinely different")
print("solutions -- direct evidence of convexity's 'every local min is global' guarantee at work.")

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. For $f(x)=-2x^2+x^4$, $x_1=-1$, $x_2=1$, $\theta=0.5$, compute both sides of the chord-test
   inequality by hand and confirm it *fails*. As a sanity check, also evaluate $f''$ at the
   blended point $\theta x_1+(1-\theta)x_2$ and confirm its sign is consistent with the
   violation. Then repeat both computations for $f(x)=x^2$ at the same $x_1,x_2,\theta$ and
   confirm the inequality holds there instead.

<details>
<summary>💡 Solution</summary>

Blended point: $\theta x_1+(1-\theta)x_2 = 0.5(-1)+0.5(1)=0$. For $f(x)=-2x^2+x^4$:
$f(0)=0$, while the chord value is $\theta f(x_1)+(1-\theta)f(x_2) = 0.5(-1)+0.5(-1)=-1$ (using
$f(\pm1)=-2+1=-1$). Since $0 > -1$, $f(\text{mix}) > \text{chord}$ — the inequality *fails*,
confirming non-convexity at this pair. Sanity check: $f''(x)=-4+12x^2$, so $f''(0)=-4<0$ —
locally concave right where the violation occurs, consistent with the failed chord test (a
convex function's chord test can never fail where $f''\ge0$). For $f(x)=x^2$ at the same points:
$f(0)=0$ and the chord value is $0.5(1)+0.5(1)=1$; since $0\le1$, the inequality holds, and in
fact $f''(x)=2>0$ everywhere, so it holds for *every* pair $(x_1,x_2,\theta)$, not just this one
— the defining difference between a globally convex function and one that merely happens to
satisfy the inequality at a particular pair.

</details>

### 💭 UNDERSTAND
2. $f(x)=x^4$ is convex (its second derivative $12x^2\ge0$ everywhere) but **not** strongly
   convex ($f''(0)=0$, so no $\mu>0$ works globally). Predict whether gradient descent on
   $f(x)=x^4$ from $x_0=1$ still achieves the Theory section's *linear* convergence — a constant
   ratio $\text{gap}_{k+1}/\text{gap}_k$ — the way it does on a genuinely strongly-convex
   quadratic. Verify by computing this ratio over several windows of a long run, for both
   $f(x)=x^4$ and a strongly-convex quadratic $f(x)=x^2$.

<details>
<summary>💡 Solution</summary>

No — strong convexity is exactly the ingredient the Theory section's linear-rate proof needs, and
without it the guarantee disappears. Verified: for $f(x)=x^2$ ($\mu=L=2$) at $\eta=0.1$, the ratio
$\text{gap}_{k+1}/\text{gap}_k$ is *exactly* $0.64$ in every window checked (steps 5-55, 100-150,
150-199) — matching $(1-\eta L)^2=(1-0.2)^2=0.64$ precisely, the hallmark of linear convergence.
For $f(x)=x^4$ at $\eta=0.05$, the same ratio is nowhere near constant: it's $\approx0.94$ over
steps 10-60, drifts to $\approx0.991$ by steps 200-250, and keeps creeping toward $1$ the longer
you run (reaching $\approx0.999$ by step 2900) — the gap is shrinking ever more *slowly* in
relative terms, the signature of a sub-linear rate (in fact $O(1/k^2)$ here, not the
$O(\rho^k)$ the Theory section guarantees for the strongly-convex case). $\mu=0$ removes the
lower bound on curvature that the proof relies on, and the practical consequence is visible
directly in the numbers: convexity alone buys you convergence, but strong convexity is what buys
you a *rate*.

</details>

### ✏️ DERIVE
3. The Theory section states the optimal constant step size is $\eta^\star=\frac{2}{L+\mu}$ for
   an $L$-smooth, $\mu$-strongly-convex $f$. For the diagonal quadratic
   $f(x)=\frac12(\mu x_1^2 + Lx_2^2)$, gradient descent's per-coordinate contraction factors are
   $|1-\eta\mu|$ and $|1-\eta L|$. Derive $\eta^\star$ by finding the $\eta$ that minimizes the
   *worst* of these two factors, $\max(|1-\eta\mu|,|1-\eta L|)$, and derive the resulting rate
   $\rho=\left(\frac{\kappa-1}{\kappa+1}\right)^2$ in terms of $\kappa=L/\mu$. Verify both by a
   numerical search (e.g. `scipy.optimize.minimize_scalar`) at $\mu=1,L=200$.

<details>
<summary>💡 Solution outline</summary>

For $\eta<1/L<1/\mu$, both factors are positive: $1-\eta\mu$ is decreasing in $\eta$, $1-\eta L$
is decreasing faster (since $L>\mu$). The two factors are equal in magnitude but opposite in sign
exactly where $1-\eta\mu = -(1-\eta L)$, i.e. $2=\eta(\mu+L)$, giving $\eta^\star=2/(L+\mu)$. This
is the minimizer of the max because to the left of this point the $\mu$-factor dominates and is
increasing as $\eta$ decreases, while to the right the $L$-factor dominates and is increasing as
$\eta$ increases — a classic equioscillation minimax argument. At $\eta^\star$: $1-\eta^\star
L = 1-\frac{2L}{L+\mu} = \frac{\mu-L}{\mu+L} = -\frac{\kappa-1}{\kappa+1}$ (dividing by $\mu$),
so the worst-case *per-step* contraction has magnitude $\frac{\kappa-1}{\kappa+1}$, and since the
loss gap involves the *squared* norm, $\rho=\left(\frac{\kappa-1}{\kappa+1}\right)^2$. Verified:
at $\mu=1,L=200$ ($\kappa=200$), `scipy.optimize.minimize_scalar` finds $\eta^\star\approx
0.0099503$, matching $2/(L+\mu)=0.0099503$ to 6 decimal places; its worst-case contraction is
$\approx0.99005=\sqrt\rho$ with $\rho\approx0.98020$, matching both the formula and this
notebook's own `gd_on_quadratic`'s empirical late-window ratio exactly.

</details>

### ✏️ DERIVE (continued)
3b. **(New) Prove the $O(1/T)$ sublinear convergence rate from the descent lemma.**
   Starting from the descent lemma with $\eta=1/L$:

   $$f(\mathbf{w}_{t+1}) \le f(\mathbf{w}_t) - \frac{1}{2L}\|\nabla f(\mathbf{w}_t)\|^2$$

   (a) Use convexity ($f(\mathbf{w}^*) \ge f(\mathbf{w}_t)+\nabla f(\mathbf{w}_t)^{\!\top}(\mathbf{w}^*-\mathbf{w}_t)$) to show $\|\nabla f(\mathbf{w}_t)\|^2 \ge \frac{(f(\mathbf{w}_t)-f^*)^2}{\|\mathbf{w}_t-\mathbf{w}^*\|^2}$ is **not** what you need — instead, show the per-step descent satisfies $f(\mathbf{w}_{t+1})-f^* \le f(\mathbf{w}_t)-f^* - \frac{1}{2L}\|\nabla f(\mathbf{w}_t)\|^2$.

   (b) Use the update rule $\mathbf{w}_{t+1}=\mathbf{w}_t-\frac{1}{L}\nabla f(\mathbf{w}_t)$ to show $\|\mathbf{w}_{t+1}-\mathbf{w}^*\|^2 \le \|\mathbf{w}_t-\mathbf{w}^*\|^2 - \frac{2}{L}(f(\mathbf{w}_t)-f^*) + \frac{1}{L^2}\|\nabla f(\mathbf{w}_t)\|^2$.

   (c) Telescope the sum $\sum_{t=0}^{T-1}$ and use the fact that $f(\mathbf{w}_T)-f^* \le f(\mathbf{w}_t)-f^*$ for all $t\le T$ (since GD is monotonically decreasing under descent lemma) to conclude $f(\mathbf{w}_T)-f^* \le \frac{L\|\mathbf{w}_0-\mathbf{w}^*\|^2}{2T}$.

<details>
<summary>💡 Solution outline</summary>

**(a)** Subtracting $f^*$ from both sides of the descent inequality: $f(\mathbf{w}_{t+1})-f^* \le (f(\mathbf{w}_t)-f^*) - \frac{1}{2L}\|\nabla f(\mathbf{w}_t)\|^2$. This is immediate.

**(b)** Expand $\|\mathbf{w}_{t+1}-\mathbf{w}^*\|^2 = \|\mathbf{w}_t - \frac{1}{L}\nabla f(\mathbf{w}_t) - \mathbf{w}^*\|^2 = \|\mathbf{w}_t-\mathbf{w}^*\|^2 - \frac{2}{L}\nabla f(\mathbf{w}_t)^{\!\top}(\mathbf{w}_t-\mathbf{w}^*) + \frac{1}{L^2}\|\nabla f(\mathbf{w}_t)\|^2$. By convexity, $\nabla f(\mathbf{w}_t)^{\!\top}(\mathbf{w}_t-\mathbf{w}^*) \ge f(\mathbf{w}_t)-f^*$, giving the bound.

**(c)** From (b), $f(\mathbf{w}_t)-f^* \le \frac{L}{2}(\|\mathbf{w}_t-\mathbf{w}^*\|^2 - \|\mathbf{w}_{t+1}-\mathbf{w}^*\|^2) + \frac{1}{2L}\|\nabla f(\mathbf{w}_t)\|^2$. Summing over $t=0,\ldots,T-1$ and using (a) to bound the gradient terms, the telescope collapses to $T(f(\mathbf{w}_T)-f^*) \le \frac{L}{2}\|\mathbf{w}_0-\mathbf{w}^*\|^2$, giving $f(\mathbf{w}_T)-f^* \le \frac{L\|\mathbf{w}_0-\mathbf{w}^*\|^2}{2T}$. $\square$

</details>

### 🐍 IMPLEMENT
4. Extend `gd_on_quadratic`: for at least 3 different $(\mu, L)$ pairs, numerically search for
   the step size that minimizes the number of steps needed to reach a fixed tolerance (rather
   than assuming $\eta^\star=2/(L+\mu)$ up front), and check that your numerically-found optimum
   matches the formula.

<details>
<summary>✅ How to know you're right</summary>

For each $(\mu,L)$, either grid-search or use `scipy.optimize.minimize_scalar` over $\eta \in
(0, 2/\mu)$ (the full stable range) to find the $\eta$ minimizing steps-to-tolerance (or,
equivalently and more cleanly, minimizing $\max(|1-\eta\mu|,|1-\eta L|)$ directly, sidestepping
tolerance-related discretization noise). Your numerically-found optimum should match
$2/(L+\mu)$ to within numerical precision — verified for $(\mu,L)=(1,10)$, $(2,50)$, and
$(0.5,30)$, each matching the formula to better than $10^{-9}$ using
`scipy.optimize.minimize_scalar` with tight tolerances. A coarse grid search will match less
precisely (a 300-point grid over $(\mu,L)=(1,10)$ lands within about $2\%$ of the true optimum) —
that's expected discretization error, not a bug, and refining the grid should visibly tighten
the match.

</details>

### 🤖 APPLY
5. A realistic feature matrix rarely has well-scaled columns — e.g. one feature $O(1)$, another
   $O(100)$ (like mixing a normalized score with a raw income figure). Build such an
   ill-conditioned linear regression problem, compute the condition number $\kappa$ of its
   Hessian $H=\frac2N X^\top X$ before and after standardizing each feature (subtracting its
   mean, dividing by its standard deviation), then run gradient descent at each version's own
   optimal $\eta^\star=2/(L+\mu)$ for a fixed step budget and compare the optimality gaps.

<details>
<summary>✅ What you should observe</summary>

Standardization should shrink $\kappa$ dramatically — in one construction (features scaled
$O(1)$ and $O(50)$), $\kappa_{\text{raw}}\approx12{,}937$ collapses to $\kappa_{\text{std}}
\approx1.16$ after standardizing, a roughly $11{,}000\times$ reduction, simply by removing the
scale mismatch between columns (standardization doesn't change the data's *information content*,
only its coordinate system). Running gradient descent at each problem's own optimal $\eta^\star$
for the same small number of steps, the standardized version's optimality gap is many orders of
magnitude smaller — after 5 steps, the raw problem's gap is still $\approx17$ while the
standardized one is already down to $\approx4\times10^{-11}$. Measuring steps-to-reach a fixed
tolerance ($10^{-4}$) makes the effect concrete: the raw problem needs on the order of $39{,}000$
steps, the standardized one only $3$ — a ratio close to the $\kappa$ reduction itself, exactly
what the Theory section's $\rho(\kappa)$ dependence predicts. This is why "standardize your
features" is a default step in almost every practical ML pipeline, not a convexity nicety: it's a
direct, controllable lever on $\kappa$, and $\kappa$ is what convexity theory says actually
determines how hard optimization is.

</details>

### 🚀 CHALLENGE
6. **(Modified)** The AI section's cell reports the *spread* (max $-$ min loss across restarts) for
   `d_hidden=6`. Investigate systematically: run the same restart experiment across several
   widths (e.g. 1, 4, 10, 20), tracking not just the spread but the *worst-case* (maximum) final
   loss across restarts at each width specifically — the same worst-case framing `04.01` used
   for its own overparameterization experiment. Does the worst-case loss shrink monotonically as
   width grows, even though non-convexity is never removed at any width? At how many restarts
   does the raw spread number become an unreliable one-run summary rather than something you'd
   trust?

7. **(New — Derive the strongly-convex rate)** Starting from the descent lemma and the strong
   convexity inequality $f(\mathbf{x})-f^* \ge \frac{\mu}{2}\|\mathbf{x}-\mathbf{w}^*\|^2$:

   (a) Show that $\|\nabla f(\mathbf{w}_t)\|^2 \ge 2\mu(f(\mathbf{w}_t)-f^*)$ (co-coercivity).

   (b) Substitute into the per-step descent inequality $f(\mathbf{w}_{t+1})-f^* \le (f(\mathbf{w}_t)-f^*) - \frac{1}{2L}\|\nabla f(\mathbf{w}_t)\|^2$ to derive $f(\mathbf{w}_{t+1})-f^* \le (1-\mu/L)(f(\mathbf{w}_t)-f^*)$.

   (c) Identify precisely where strong convexity enters the proof (step (a)) and explain why the
   argument fails for merely convex $f$.

   (d) Verify your derived rate numerically: run GD on a quadratic with $\mu=1, L=50$,
   measure the empirical per-step ratio, and check it matches $\rho=1-1/\kappa=0.98$.

<details>
<summary>🔍 What a strong answer covers</summary>

**(a)** From strong convexity: $f(\mathbf{w}^*) \ge f(\mathbf{w}_t) + \nabla f(\mathbf{w}_t)^{\!\top}(\mathbf{w}^*-\mathbf{w}_t) + \frac{\mu}{2}\|\mathbf{w}^*-\mathbf{w}_t\|^2$. Rearranging: $\nabla f(\mathbf{w}_t)^{\!\top}(\mathbf{w}_t-\mathbf{w}^*) \ge f(\mathbf{w}_t)-f^* + \frac{\mu}{2}\|\mathbf{w}_t-\mathbf{w}^*\|^2$. Combined with the co-coercivity lemma (from $L$-smoothness): $\|\nabla f(\mathbf{w}_t)\|^2 \ge 2\mu(f(\mathbf{w}_t)-f^*)$.

**(b)** Direct substitution: $f(\mathbf{w}_{t+1})-f^* \le (f(\mathbf{w}_t)-f^*) - \frac{1}{2L}\cdot 2\mu(f(\mathbf{w}_t)-f^*) = (1-\mu/L)(f(\mathbf{w}_t)-f^*)$.

**(c)** Strong convexity enters in (a) to convert a distance bound ($\|\mathbf{w}_t-\mathbf{w}^*\|^2$) into a function-value bound ($f(\mathbf{w}_t)-f^*$). For merely convex $f$, $\mu=0$ and the inequality $\|\nabla f\|^2 \ge 0$ is vacuous — you cannot get a *multiplicative* contraction, only the additive telescoping of Exercise 3b.

**(d)** Empirically, GD on $f(x)=0.5(x_1^2 + 50\,x_2^2)$ with $\eta=1/50$ gives a late-window ratio of $\approx0.98$, matching $1-1/50=0.98$ exactly.

</details>

---

## 📚 Further Reading
- Boyd & Vandenberghe, [*Convex Optimization*](https://web.stanford.edu/~boyd/cvxbook/), Ch. 2–3
- Nesterov, *Introductory Lectures on Convex Optimization: A Basic Course* (2004), Ch. 2
- Bubeck, ["Convex Optimization: Algorithms and Complexity"](https://arxiv.org/abs/1405.4980)

---

*Next notebook: [Momentum and Adaptive Methods](04_momentum_and_adaptive_methods.ipynb)*